In [ ]:
%sql
-- Databricks SQL script: Comprehensive test suite for patient_data table field addition and validation
-- Purpose: Validate schema, data types, constraints, ingestion, error handling, Delta Lake operations, window analytics, and final output display for patient_data table
-- Author: Giang Nguyen
-- Date: 2025-10-13
-- Description: This script tests the addition and validation of new fields in the patient_data table, including schema checks, data type conversions, constraint enforcement, error handling, Delta Lake operations, window analytics, and final output display. It covers unit, integration, and data quality tests.

USE CATALOG purgo_databricks;

-- =========================
-- 1. SCHEMA VALIDATION TESTS
-- =========================

-- Validate that patient_data table contains all required columns with correct data types and nullability

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS missing_columns
FROM required_columns rc
LEFT JOIN actual_columns ac
  ON rc.column_name = ac.column_name
  AND rc.data_type = ac.data_type
  AND rc.is_nullable = ac.is_nullable
WHERE ac.column_name IS NULL;
-- Assert: missing_columns = 0 (all required columns present)

-- =========================
-- 2. CONSTRAINT VALIDATION TESTS
-- =========================

-- Validate record_status allowed values using CHECK constraint

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS invalid_status_count
FROM purgo_playground.patient_data
WHERE record_status IS NOT NULL
  AND record_status NOT IN ("active", "inactive", "pending");
-- Assert: invalid_status_count = 0

-- =========================
-- 3. DATA TYPE CONVERSION & NULL HANDLING TESTS
-- =========================

-- Test ingestion_date is either valid timestamp or NULL

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS invalid_ingestion_date_count
FROM purgo_playground.patient_data
WHERE ingestion_date IS NOT NULL
  AND TRY_CAST(ingestion_date AS TIMESTAMP) IS NULL;
-- Assert: invalid_ingestion_date_count = 0

-- Test source_system is non-empty string or NULL

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS invalid_source_system_count
FROM purgo_playground.patient_data
WHERE source_system IS NOT NULL AND TRIM(source_system) = "";
-- Assert: invalid_source_system_count = 0

-- Test created_by is non-empty string or NULL

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS invalid_created_by_count
FROM purgo_playground.patient_data
WHERE created_by IS NOT NULL AND TRIM(created_by) = "";
-- Assert: invalid_created_by_count = 0

-- Test modified_by is non-empty string or NULL

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS invalid_modified_by_count
FROM purgo_playground.patient_data
WHERE modified_by IS NOT NULL AND TRIM(modified_by) = "";
-- Assert: invalid_modified_by_count = 0

-- =========================
-- 4. DUPLICATE PRIMARY KEY VALIDATION
-- =========================

-- Check for duplicate patient_id values

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS duplicate_patient_id_count
FROM (

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT patient_id
  FROM purgo_playground.patient_data
  WHERE patient_id IS NOT NULL
  GROUP BY patient_id
  HAVING COUNT(*) > 1
) dupes;
-- Assert: duplicate_patient_id_count = 0

-- =========================
-- 5. DELTA LAKE OPERATIONS TESTS
-- =========================

-- Test MERGE: Simulate upsert of new patient record with all new fields

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

MERGE INTO purgo_playground.patient_data AS target
USING (

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT
    1011 AS patient_id,
    "Test User" AS patient_name,
    29 AS age,
    "Test Diagnosis" AS diagnosis,
    "Test Treatment" AS treatment,
    CAST("2024-06-08T12:00:00" AS TIMESTAMP) AS ingestion_date,
    "EMR" AS source_system,
    "active" AS record_status,
    "tester" AS created_by,
    "tester" AS modified_by
) AS src
ON target.patient_id = src.patient_id
WHEN MATCHED THEN
  UPDATE SET
    patient_name = src.patient_name,
    age = src.age,
    diagnosis = src.diagnosis,
    treatment = src.treatment,
    ingestion_date = src.ingestion_date,
    source_system = src.source_system,
    record_status = src.record_status,
    created_by = src.created_by,
    modified_by = src.modified_by
WHEN NOT MATCHED THEN
  INSERT (
    patient_id, patient_name, age, diagnosis, treatment,
    ingestion_date, source_system, record_status, created_by, modified_by
  )
  VALUES (
    src.patient_id, src.patient_name, src.age, src.diagnosis, src.treatment,
    src.ingestion_date, src.source_system, src.record_status, src.created_by, src.modified_by
  );
-- Assert: patient_id 1011 exists and fields match

-- Test DELETE: Remove test record

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

DELETE FROM purgo_playground.patient_data
WHERE patient_id = 1011;
-- Assert: patient_id 1011 no longer exists

-- =========================
-- 6. WINDOW FUNCTION & ANALYTICS TESTS
-- =========================

-- Use window function to get most recent ingestion_date per source_system

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT patient_id, source_system, ingestion_date
FROM (

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT
    patient_id,
    source_system,
    ingestion_date,
    ROW_NUMBER() OVER (PARTITION BY source_system ORDER BY ingestion_date DESC) AS rn
  FROM purgo_playground.patient_data
) ranked_patients
WHERE rn = 1;

-- =========================
-- 7. BACKFILL LOGIC VALIDATION
-- =========================

-- Simulate backfill for NULL fields using patient_hist and default values
-- (This is a validation query, not an update)
WITH hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
)

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT
  pd.patient_id,
  COALESCE(pd.ingestion_date, hd.min_hist_date) AS backfilled_ingestion_date,
  COALESCE(pd.source_system, "Unknown") AS backfilled_source_system,
  COALESCE(pd.record_status, "active") AS backfilled_record_status,
  COALESCE(pd.created_by, "system") AS backfilled_created_by,
  COALESCE(pd.modified_by, "system") AS backfilled_modified_by
FROM purgo_playground.patient_data pd
LEFT JOIN hist_dates hd ON pd.patient_id = hd.patient_id
WHERE pd.ingestion_date IS NULL
   OR pd.source_system IS NULL
   OR pd.record_status IS NULL
   OR pd.created_by IS NULL
   OR pd.modified_by IS NULL;

-- =========================
-- 8. INTEGRATION TEST: FULL INGESTION VALIDATION
-- =========================

-- Validate that all test data rows are present and correct

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT
  e.patient_id,
  CASE WHEN pd.patient_id IS NULL THEN "Missing" ELSE "Present" END AS status
FROM expected e
LEFT JOIN purgo_playground.patient_data pd
  ON e.patient_id = pd.patient_id;
-- Assert: All status = "Present"

-- =========================
-- 9. PERFORMANCE TEST: INGESTION COUNT
-- =========================

-- Check that patient_data table row count matches expected test data count (should be >= 5)

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT COUNT(*) AS patient_data_row_count
FROM purgo_playground.patient_data;

-- =========================
-- 10. FINAL OUTPUT DISPLAY
-- =========================

-- Display the updated patient_data table with all required columns

WITH required_columns AS (
  SELECT "patient_id" AS column_name, "bigint" AS data_type, "YES" AS is_nullable
  UNION ALL SELECT "patient_name", "string", "YES"
  UNION ALL SELECT "age", "bigint", "YES"
  UNION ALL SELECT "diagnosis", "string", "YES"
  UNION ALL SELECT "treatment", "string", "YES"
  UNION ALL SELECT "ingestion_date", "timestamp", "YES"
  UNION ALL SELECT "source_system", "string", "YES"
  UNION ALL SELECT "record_status", "string", "YES"
  UNION ALL SELECT "created_by", "string", "YES"
  UNION ALL SELECT "modified_by", "string", "YES"
),
actual_columns AS (
  SELECT
    column_name,
    data_type,
    is_nullable
  FROM information_schema.columns
  WHERE table_catalog = "purgo_databricks"
    AND table_schema = "purgo_playground"
    AND table_name = "patient_data"
),
hist_dates AS (
  SELECT
    CAST(patient_sf_id AS BIGINT) AS patient_id,
    MIN(CAST(created_date AS TIMESTAMP)) AS min_hist_date
  FROM purgo_playground.patient_hist
  WHERE patient_sf_id IS NOT NULL
  GROUP BY patient_sf_id
),
expected AS (
  SELECT
    1001 AS patient_id, "John Smith" AS patient_name, 45 AS age, "Diabetes" AS diagnosis, "Insulin" AS treatment,
    CAST("2024-06-01T10:00:00" AS TIMESTAMP) AS ingestion_date, "EMR" AS source_system, "active" AS record_status, "admin" AS created_by, "admin" AS modified_by
  UNION ALL
  SELECT 1002, "Jane Doe", 30, "Asthma", "Inhaler", CAST("2024-06-02T11:00:00" AS TIMESTAMP), "Portal", "pending", "nurse", "nurse"
  UNION ALL
  SELECT 1003, "Bob Lee", 60, "Cancer", "Chemo", NULL, "EMR", "inactive", "doctor", "doctor"
  UNION ALL
  SELECT 1004, "Alice Kim", 50, "Hypertension", "Beta Blocker", CAST("2024-06-03T09:30:00" AS TIMESTAMP), NULL, "active", "admin", "nurse"
  UNION ALL
  SELECT 1005, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
)

SELECT
  patient_id,
  patient_name,
  age,
  diagnosis,
  treatment,
  ingestion_date,
  source_system,
  record_status,
  created_by,
  modified_by
FROM purgo_playground.patient_data
ORDER BY patient_id;

-- =========================
-- 11. CLEANUP OPERATIONS
-- =========================

-- No temp views or temp tables used; no cleanup required

-- END OF SCRIPT
